<a href="https://colab.research.google.com/github/UntalHugo/universidad-ia/blob/main/clase-01-rag-ventas/01_normalizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗂️ Paso A — Exploración del dataset crudo
Objetivo: entender la estructura del CSV antes de normalizar.

# Celda 1 — Instalación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# No necesitamos nada especial, pandas ya viene en Colab
import pandas as pd
import numpy as np

print("✅ Librerías listas")

✅ Librerías listas


#Celda 2 — Cargar el CSV


In [ ]:
# El encoding latin-1 es necesario porque el archivo tiene caracteres especiales
# como tildes o la ñ guardados en formato antiguo (no UTF-8)
df = pd.read_csv("sales_data_sample.csv", encoding="latin-1")

print(f"✅ Dataset cargado")
print(f"   Filas    : {df.shape[0]}")
print(f"   Columnas : {df.shape[1]}")

✅ Dataset cargado
   Filas    : 2823
   Columnas : 25


#Celda 3 — Ver las primeras filas


In [ ]:
# .head() muestra las primeras 5 filas
# Sirve para ver cómo se ve la data "en crudo"
df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


#Celda 4 — Tipos de datos y nulos


In [ ]:
print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print()
print("=== VALORES NULOS POR COLUMNA ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0])  # Solo muestra las que tienen nulos

=== TIPOS DE DATOS ===
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object

=== VALORES NULOS POR COLUMNA ===
ADDRESSLINE2    2521
STATE           1486
POSTALCODE        76
TERRITORY       1074
dtype: int64


#Celda 5 — El diagnóstico del problema (la parte clave)


In [ ]:
print("=== ¿CUÁNTOS REGISTROS ÚNICOS HAY REALMENTE? ===")
print(f"Total de filas     : {len(df)}")
print(f"Órdenes únicas     : {df['ORDERNUMBER'].nunique()}")
print(f"Clientes únicos    : {df['CUSTOMERNAME'].nunique()}")
print(f"Productos únicos   : {df['PRODUCTCODE'].nunique()}")
print(f"Líneas de orden    : {len(df)}")  # cada fila ES una línea de orden
print()
print("=== CONCLUSIÓN ===")
print(f"El dataset repite datos de {df['CUSTOMERNAME'].nunique()} clientes")
print(f"en {len(df)} filas. Eso es desnormalización.")

=== ¿CUÁNTOS REGISTROS ÚNICOS HAY REALMENTE? ===
Total de filas     : 2823
Órdenes únicas     : 307
Clientes únicos    : 92
Productos únicos   : 109
Líneas de orden    : 2823

=== CONCLUSIÓN ===
El dataset repite datos de 92 clientes
en 2823 filas. Eso es desnormalización.


#Celda 6 — Visualizar la repetición (la prueba)

In [ ]:
# Tomamos un cliente cualquiera y vemos cuántas veces se repite su info
cliente = "Land of Toys Inc."
columnas_cliente = ["ORDERNUMBER", "CUSTOMERNAME", "PHONE", "ADDRESSLINE1", "CITY", "COUNTRY"]

print(f"=== Apariciones de '{cliente}' ===")
print(df[df["CUSTOMERNAME"] == cliente][columnas_cliente].to_string())

=== Apariciones de 'Land of Toys Inc.' ===
      ORDERNUMBER       CUSTOMERNAME       PHONE             ADDRESSLINE1 CITY COUNTRY
0           10107  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
19          10329  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
54          10107  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
73          10329  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
80          10107  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
99          10329  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
117         10248  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
180         10329  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
240         10107  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA
259         10329  Land of Toys Inc.  2125557818  897 Long Airport Avenue  NYC     USA


#Celda 7 — Resumen final del diagnóstico

In [ ]:
print("=== 📋 DIAGNÓSTICO COMPLETO ===")
print()
print("Tablas que deberíamos tener (forma normalizada):")
print(f"  dim_clientes    →  {df['CUSTOMERNAME'].nunique()} filas")
print(f"  dim_productos   →  {df['PRODUCTCODE'].nunique()} filas")
print(f"  fact_ordenes    →  {df['ORDERNUMBER'].nunique()} filas")
print(f"  fact_items      →  {len(df)} filas  ← esta conserva las 2823 filas")
print()
print("Columnas por tabla:")
print("  dim_clientes  : CUSTOMERNAME, PHONE, ADDRESS, CITY, STATE,")
print("                  POSTALCODE, COUNTRY, TERRITORY, CONTACTFIRST, CONTACTLAST")
print("  dim_productos : PRODUCTCODE, PRODUCTLINE, MSRP")
print("  fact_ordenes  : ORDERNUMBER, ORDERDATE, STATUS, QTR_ID,")
print("                  MONTH_ID, YEAR_ID, DEALSIZE, CUSTOMERNAME")
print("  fact_items    : ORDERNUMBER, ORDERLINENUMBER, PRODUCTCODE,")
print("                  QUANTITYORDERED, PRICEEACH, SALES")

=== 📋 DIAGNÓSTICO COMPLETO ===

Tablas que deberíamos tener (forma normalizada):
  dim_clientes    →  92 filas
  dim_productos   →  109 filas
  fact_ordenes    →  307 filas
  fact_items      →  2823 filas  ← esta conserva las 2823 filas

Columnas por tabla:
  dim_clientes  : CUSTOMERNAME, PHONE, ADDRESS, CITY, STATE,
                  POSTALCODE, COUNTRY, TERRITORY, CONTACTFIRST, CONTACTLAST
  dim_productos : PRODUCTCODE, PRODUCTLINE, MSRP
  fact_ordenes  : ORDERNUMBER, ORDERDATE, STATUS, QTR_ID,
                  MONTH_ID, YEAR_ID, DEALSIZE, CUSTOMERNAME
  fact_items    : ORDERNUMBER, ORDERLINENUMBER, PRODUCTCODE,
                  QUANTITYORDERED, PRICEEACH, SALES


# 🔧 Paso B — Normalización del dataset
Vamos a dividir las 25 columnas en 4 tablas limpias.

Celda 8 — dim_clientes


In [ ]:
# ¿Qué hace esta celda?
# Extrae toda la información del cliente en una sola tabla.
# .drop_duplicates() elimina las filas repetidas — si el mismo cliente
# aparece 30 veces en el CSV, acá va a quedar UNA sola fila.

dim_clientes = df[[
    "CUSTOMERNAME",
    "PHONE",
    "ADDRESSLINE1",
    "ADDRESSLINE2",
    "CITY",
    "STATE",
    "POSTALCODE",
    "COUNTRY",
    "TERRITORY",
    "CONTACTLASTNAME",
    "CONTACTFIRSTNAME"
]].drop_duplicates()

# Tratamiento de NaN para columnas de texto (como hablamos)
dim_clientes["ADDRESSLINE2"] = dim_clientes["ADDRESSLINE2"].fillna("")
dim_clientes["STATE"]        = dim_clientes["STATE"].fillna("N/A")
dim_clientes["POSTALCODE"]   = dim_clientes["POSTALCODE"].fillna("N/A")
dim_clientes["TERRITORY"]    = dim_clientes["TERRITORY"].fillna("Sin asignar")

# Creamos un ID numérico único para cada cliente
# reset_index() reinicia el índice después del drop_duplicates
dim_clientes = dim_clientes.reset_index(drop=True)
dim_clientes.insert(0, "CUSTOMER_ID", dim_clientes.index + 1)

print(f"✅ dim_clientes: {dim_clientes.shape[0]} filas, {dim_clientes.shape[1]} columnas")
dim_clientes.head()

✅ dim_clientes: 92 filas, 12 columnas


,CUSTOMER_ID,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME
0,1,Land of Toys Inc.,2125557818,897 Long Airport Avenue,,NYC,NY,10022,USA,Sin asignar,Yu,Kwai
1,2,Reims Collectables,26.47.1555,59 rue de l'Abbaye,,Reims,N/A,51100,France,EMEA,Henriot,Paul
2,3,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,,Paris,N/A,75508,France,EMEA,Da Cunha,Daniel
3,4,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,,Pasadena,CA,90003,USA,Sin asignar,Young,Julie
4,5,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,,San Francisco,CA,N/A,USA,Sin asignar,Brown,Julie


Celda 9 — dim_productos


In [ ]:
# Misma lógica: extraemos solo las columnas del producto
# y eliminamos duplicados. Cada producto tiene su propio código (PRODUCTCODE)
# que ya funciona como ID natural — no necesitamos crear uno nuevo.

dim_productos = df[[
    "PRODUCTCODE",
    "PRODUCTLINE",
    "MSRP"           # precio de lista sugerido por el fabricante
]].drop_duplicates()

dim_productos = dim_productos.reset_index(drop=True)

print(f"✅ dim_productos: {dim_productos.shape[0]} filas, {dim_productos.shape[1]} columnas")
dim_productos.head()

✅ dim_productos: 109 filas, 3 columnas


,PRODUCTCODE,PRODUCTLINE,MSRP
0,S10_1678,Motorcycles,95
1,S10_1949,Classic Cars,214
2,S10_2016,Motorcycles,118
3,S10_4698,Motorcycles,193
4,S10_4757,Classic Cars,136


Celda 10 — Paso previo para fact_ordenes (unir el CUSTOMER_ID)


In [ ]:
# Antes de crear fact_ordenes necesitamos el CUSTOMER_ID que generamos
# en dim_clientes, porque fact_ordenes va a guardar la FK (foreign key)
# en vez de repetir todo el nombre y dirección del cliente.

# Hacemos un merge: unimos el df original con dim_clientes
# buscando coincidencia por CUSTOMERNAME, y traemos solo el CUSTOMER_ID

df_con_id = df.merge(
    dim_clientes[["CUSTOMER_ID", "CUSTOMERNAME"]],
    on="CUSTOMERNAME",
    how="left"
)

print(f"✅ Merge hecho. Filas: {df_con_id.shape[0]}")
print(f"   CUSTOMER_ID asignados: {df_con_id['CUSTOMER_ID'].notna().sum()}")

✅ Merge hecho. Filas: 2823
   CUSTOMER_ID asignados: 2823


Celda 11 — fact_ordenes


In [ ]:
fact_ordenes = df_con_id[[
    "ORDERNUMBER",
    "ORDERDATE",
    "STATUS",
    "QTR_ID",
    "MONTH_ID",
    "YEAR_ID",
    "DEALSIZE",
    "CUSTOMER_ID"
]].drop_duplicates(subset=["ORDERNUMBER"])  # ← forzamos unicidad por ORDERNUMBER

fact_ordenes["ORDERDATE"] = pd.to_datetime(fact_ordenes["ORDERDATE"])
fact_ordenes = fact_ordenes.reset_index(drop=True)

print(f"✅ fact_ordenes: {fact_ordenes.shape[0]} filas")

✅ fact_ordenes: 307 filas


Celda 12 — fact_items


In [ ]:
# Esta es la tabla de hechos principal — conserva las 2823 filas originales.
# Cada fila es una línea de orden: qué producto, cuántas unidades, a qué precio.
# No repetimos datos del cliente ni del producto, solo sus IDs (FK).

fact_items = df[[
    "ORDERNUMBER",        # FK → fact_ordenes
    "ORDERLINENUMBER",    # número de línea dentro de la orden
    "PRODUCTCODE",        # FK → dim_productos
    "QUANTITYORDERED",    # cuántas unidades
    "PRICEEACH",          # precio unitario real (puede diferir del MSRP)
    "SALES"               # total de la línea = QUANTITYORDERED * PRICEEACH
]]

fact_items = fact_items.reset_index(drop=True)

print(f"✅ fact_items: {fact_items.shape[0]} filas, {fact_items.shape[1]} columnas")
fact_items.head()

✅ fact_items: 2823 filas, 6 columnas


,ORDERNUMBER,ORDERLINENUMBER,PRODUCTCODE,QUANTITYORDERED,PRICEEACH,SALES
0,10107,2,S10_1678,30,95.70,2871.00
1,10121,5,S10_1678,34,81.35,2765.90
2,10134,2,S10_1678,41,94.74,3884.34
3,10145,6,S10_1678,45,83.26,3746.70
4,10159,14,S10_1678,49,100.00,5205.27


Celda 13 — Verificación final

In [ ]:
print("=" * 45)
print("  RESUMEN DE NORMALIZACIÓN")
print("=" * 45)
print(f"  CSV original  → {df.shape[0]:>5} filas  {df.shape[1]:>3} columnas")
print("-" * 45)
print(f"  dim_clientes  → {dim_clientes.shape[0]:>5} filas  {dim_clientes.shape[1]:>3} columnas")
print(f"  dim_productos → {dim_productos.shape[0]:>5} filas  {dim_productos.shape[1]:>3} columnas")
print(f"  fact_ordenes  → {fact_ordenes.shape[0]:>5} filas  {fact_ordenes.shape[1]:>3} columnas")
print(f"  fact_items    → {fact_items.shape[0]:>5} filas  {fact_items.shape[1]:>3} columnas")
print("=" * 45)

  RESUMEN DE NORMALIZACIÓN
  CSV original  →  2823 filas   25 columnas
---------------------------------------------
  dim_clientes  →    92 filas   12 columnas
  dim_productos →   109 filas    3 columnas
  fact_ordenes  →   698 filas    8 columnas
  fact_items    →  2823 filas    6 columnas


Celda 14 — Exportar los 4 CSVs limpios

In [ ]:
# Guardamos cada tabla como CSV separado.
# index=False evita que pandas agregue una columna extra con el número de fila.

dim_clientes.to_csv("dim_clientes.csv",   index=False)
dim_productos.to_csv("dim_productos.csv", index=False)
fact_ordenes.to_csv("fact_ordenes.csv",   index=False)
fact_items.to_csv("fact_items.csv",       index=False)

print("✅ 4 archivos exportados:")
print("   dim_clientes.csv")
print("   dim_productos.csv")
print("   fact_ordenes.csv")
print("   fact_items.csv")
print()
print("📥 Descargalos desde el panel de archivos (izquierda)")

✅ 4 archivos exportados:
   dim_clientes.csv
   dim_productos.csv
   fact_ordenes.csv
   fact_items.csv

📥 Descargalos desde el panel de archivos (izquierda)
